# 01 — Environment check

Verify packages, data path, and (on Colab) GPU before training.

| Where | Kernel / runtime | Expect CUDA? |
|-------|------------------|--------------|
| **Cursor (Nate, local)** | `COMP9517-DL-Nate` (`comp9517-dl-nate`), Python 3.12 | **No** — Intel Arc only; CPU inspect / path checks |
| **Google Colab (graded runs)** | Runtime → GPU (T4) | **Yes** — required for training / eval / Grad-CAM |
| Abdoali (RTX 2050) | `COMP9517-DL` | Yes |

Set `INAT_DATA_ROOT` before importing `src.config` if your subset is not at repo-root `subset/`.

## 0. Colab bootstrap (skip on local Cursor)

Run these cells **only in Google Colab**. Upload `subset/` (~2.5 GB) to Drive once first.

In [1]:
# Colab bootstrap — mounts Drive, sets INAT_DATA_ROOT, installs deps.
# Safe no-op on local Cursor.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"IN_COLAB = {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_SUBSET = Path("/content/drive/MyDrive/comp9517/subset")
    REPO_DIR = Path("/content/Haramcomp9517")
    BRANCH = "nate/pretrained-gradcam"
    REPO_URL = "https://github.com/NateRebello/Haramcomp9517.git"

    assert DRIVE_SUBSET.is_dir(), f"Missing {DRIVE_SUBSET} — check Drive path / shortcut"
    for split in ("train", "val", "test"):
        assert (DRIVE_SUBSET / split).is_dir(), f"Missing {DRIVE_SUBSET / split}"

    if not REPO_DIR.is_dir():
        # Fall back to main/deepl branch if Nate's branch is not on remote yet
        try:
            subprocess.check_call(
                ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)]
            )
        except subprocess.CalledProcessError:
            print(f"Branch {BRANCH} not on remote — cloning default branch")
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "fetch", "origin"], cwd=REPO_DIR)

    pipeline = REPO_DIR / "dl_pipeline"
    # Prefer the uploaded notebook's sibling tree if present; else cloned repo
    os.chdir(pipeline)
    os.environ["INAT_DATA_ROOT"] = str(DRIVE_SUBSET)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    if str(pipeline) not in sys.path:
        sys.path.insert(0, str(pipeline))
    print("cwd =", os.getcwd())
    print("INAT_DATA_ROOT =", os.environ["INAT_DATA_ROOT"])
else:
    # Local: default DATA_ROOT is repo-root/subset via src.config
    print("Local run — using default DATA_ROOT unless INAT_DATA_ROOT is set.")

# Local Abdoali path lock (before src.config import)
import os
os.environ.setdefault(
    "INAT_DATA_ROOT",
    r"C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset",
)
print("INAT_DATA_ROOT =", os.environ["INAT_DATA_ROOT"])


IN_COLAB = False
Local run — using default DATA_ROOT unless INAT_DATA_ROOT is set.
INAT_DATA_ROOT = C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset


## 1. Project constants

Import shared paths from `src.config` (seed 42, portable `DATA_ROOT`).
Set `INAT_DATA_ROOT` in the Colab cell above *before* this import if needed.


In [2]:
import sys
from pathlib import Path

# Make dl_pipeline importable whether cwd is notebooks/ or dl_pipeline/
_here = Path.cwd().resolve()
for _root in (_here, _here.parent):
    if (_root / "src" / "config.py").is_file():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break

from src.config import (
    SEED,
    PROJECT_ROOT,
    DATA_ROOT,
    CHECKPOINTS_DIR,
    RESULTS_DIR,
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    ensure_output_dirs,
)

ensure_output_dirs()
SRC_DIR = PROJECT_ROOT / "src"

print(f"SEED          = {SEED}")
print(f"PROJECT_ROOT  = {PROJECT_ROOT}")
print(f"DATA_ROOT     = {DATA_ROOT}  exists={DATA_ROOT.is_dir()}")
print(
    f"checkpoints   = {CHECKPOINTS_DIR.is_dir()}, "
    f"results = {RESULTS_DIR.is_dir()}, src = {SRC_DIR.is_dir()}"
)
for split, p in (("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)):
    n_classes = len([d for d in p.iterdir() if d.is_dir()]) if p.is_dir() else 0
    print(f"  {split}: {p}  class_folders={n_classes}")
    assert p.is_dir() and n_classes == 500, f"Expected 500 class folders under {p}"


c:\Users\Abdoali\Comp9517\Group_project\dl_pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SEED          = 42
PROJECT_ROOT  = C:\Users\Abdoali\Comp9517\Group_project\Haramcomp9517\dl_pipeline
DATA_ROOT     = C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset  exists=True
checkpoints   = True, results = True, src = True
  train: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\train  class_folders=500
  val: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\val  class_folders=500
  test: C:\Users\Abdoali\Comp9517\Group_project\inat_subset\subset\test  class_folders=500


## 2. Package versions

Record these in the report's experimental-setup section.

In [3]:
import sys
import platform

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns
import tqdm
import torch
import torchvision

print(f"Python        : {sys.version.split()[0]}  ({platform.platform()})")
print(f"torch         : {torch.__version__}")
print(f"torchvision   : {torchvision.__version__}")
print(f"numpy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"matplotlib    : {matplotlib.__version__}")
print(f"seaborn       : {sns.__version__}")
print(f"tqdm          : {tqdm.__version__}")

Python        : 3.12.10  (Windows-11-10.0.26200-SP0)
torch         : 2.13.0+cu126
torchvision   : 0.28.0+cu126
numpy         : 2.4.4
pandas        : 3.0.5
scikit-learn  : 1.9.0
matplotlib    : 3.11.1
seaborn       : 0.13.2
tqdm          : 4.70.0


## 3. GPU / CUDA visibility

- **On CUDA machines, prefer mixed precision and batch size 16 for ResNet-18.


In [4]:
print(f"cuda.is_available() : {torch.cuda.is_available()}")
print(f"torch.version.cuda  : {torch.version.cuda}")
print(f"cuDNN enabled       : {torch.backends.cudnn.enabled}")

if torch.cuda.is_available():
    idx = 0
    props = torch.cuda.get_device_properties(idx)
    total_gb = props.total_memory / (1024 ** 3)
    print(f"device name         : {torch.cuda.get_device_name(idx)}")
    print(f"compute capability  : {torch.cuda.get_device_capability(idx)}")
    print(f"total VRAM          : {total_gb:.2f} GiB")
    print(f"multi-processor cnt : {props.multi_processor_count}")
else:
    print(
        "CUDA not available on this runtime.\n"
        "  - Local Nate: expected (CPU torch / no NVIDIA). Continue path checks; train on Colab.\n"
        "  - Colab: Runtime → Change runtime type → GPU (T4), then re-run this notebook."
    )


cuda.is_available() : True
torch.version.cuda  : 12.6
cuDNN enabled       : True
device name         : NVIDIA GeForce RTX 2050
compute capability  : (8, 6)
total VRAM          : 4.00 GiB
multi-processor cnt : 16


## 4. Tiny device smoke test

Runs a small matmul on CUDA if available, otherwise on CPU (local inspect path).


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"smoke device = {device}")
a = torch.randn(1024, 1024, device=device)
b = torch.randn(1024, 1024, device=device)
c = a @ b
print(f"matmul result sum   : {c.sum().item():.4f}")
if device.type == "cuda":
    alloc_mb = torch.cuda.memory_allocated(device) / (1024 ** 2)
    print(f"memory allocated    : {alloc_mb:.1f} MiB")
print("device smoke test OK")


smoke device = cuda
matmul result sum   : -6562.8579
memory allocated    : 20.1 MiB
device smoke test OK


## 5. Mixed-precision sanity check

We will use AMP by default in training notebooks to stretch 4GB VRAM. This cell only confirms autocast runs.

In [6]:
from torch.amp import autocast

x = torch.randn(32, 3, 64, 64, device=device)
with autocast("cuda", dtype=torch.float16):
    y = torch.nn.functional.conv2d(x, torch.randn(8, 3, 3, 3, device=device))
print(f"AMP output dtype    : {y.dtype}")  # expect torch.float16
print("AMP smoke test OK")
del x, y
torch.cuda.empty_cache()

AMP output dtype    : torch.float16
AMP smoke test OK
